## Installation

### Prerequisites

```bash
pip install kaggle kagglehub pyyaml
```

### Kaggle Authentication

You need Kaggle API credentials to download datasets.

#### Option 1: Local Development (kaggle.json)

1. Go to [Kaggle Account Settings](https://www.kaggle.com/settings/account)
2. Scroll to "API" section
3. Click "Create New Token"
4. Download `kaggle.json`
5. Place it in the correct location:

**Linux/macOS:**
```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
```

**Windows:**
```powershell
# Place kaggle.json in:
C:\Users\<YourUsername>\.kaggle\kaggle.json
```

#### Option 2: RunPod/Docker (Environment Variables)

Set environment variables in your RunPod/Docker container:

```bash
export KAGGLE_USERNAME="your_username"
export KAGGLE_KEY="your_api_key"
```

Or add to your Dockerfile:
```dockerfile
ENV KAGGLE_USERNAME=your_username
ENV KAGGLE_KEY=your_api_key
```

### kaggle.json Locations Reference

| Platform | Location |
|----------|----------|
| **Linux** | `~/.kaggle/kaggle.json` or `~/.config/kaggle/kaggle.json` |
| **macOS** | `~/.kaggle/kaggle.json` |
| **Windows** | `C:\Users\<YourUsername>\.kaggle\kaggle.json` |
| **WSL** | `/home/<username>/.kaggle/kaggle.json` |

---

## Configuration

Edit `config/train.yaml`:

```yaml
data:
  kaggle_dataset: "owner/dataset-name"  # e.g., "masoudnickparvar/brain-tumor-mri-dataset"
  path: "./data/train"                  # Relative to kaggle_structure directory
```

---


---

## Finding Kaggle Datasets

### Method 1: Kaggle Website

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets)
2. Search for your topic with relevant keywords:
   - **Medical:** "Brain MRI", "Tumor", "Cancer", "Lung", "Histopathology", "medical imaging", "lesion"
   - **Computer Vision:** "object detection", "segmentation", "saliency", "image similarity"
   - **3D Vision:** "LiDAR", "SLAM", "Photogrammetry", "stereo", "monocular depth"
3. Click on a dataset
4. The dataset name is in the URL: `kaggle.com/datasets/OWNER/DATASET-NAME`
5. Use `OWNER/DATASET-NAME` in your `config/train.yaml`

**Example searches:**
- "Brain Tumor MRI" → Brain tumor detection datasets
- "lung cancer CT" → Lung cancer imaging datasets
- "object detection COCO" → Object detection datasets
- "LiDAR point cloud" → 3D LiDAR datasets
- "histopathology cancer" → Medical histopathology images
- "semantic segmentation" → Segmentation datasets
- "SLAM dataset" → Robotics SLAM datasets

### Method 2: Kaggle CLI

The Kaggle CLI is a powerful tool for searching and downloading datasets from the command line.


#### Searching Datasets

**Basic search:**
```bash
# Search by keyword
kaggle datasets list -s "MRI"
kaggle datasets list -s "Brain Tumor"
kaggle datasets list -s "object detection"
```

**Sort options:**

Available sort methods: `hottest`, `votes`, `updated`, `active`, `published`

```bash
# Sort by votes (most popular)
kaggle datasets list -s "MRI" --sort-by votes

# Sort by recently updated
kaggle datasets list -s "Brain Tumor" --sort-by updated

# Sort by most active (recently discussed)
kaggle datasets list -s "lung cancer" --sort-by active

# Sort by hottest (trending)
kaggle datasets list -s "histopathology" --sort-by hottest
```

**Pagination:**

Use `-p` or `--page` to see more results:

```bash
# Get page 3 with specific sorting
kaggle datasets list -s "medical imaging" --sort-by votes -p 3
```

**More search options:**

```bash
# Show more results per page (max 100)
kaggle datasets list -s "segmentation" --max-size 50

# Search by size (small, medium, large)
kaggle datasets list -s "brain" --file-type csv

# Search by license
kaggle datasets list -s "MRI" --license cc
```

**Output format:**

```bash
# Example output:
# masoudnickparvar/brain-tumor-mri-dataset
# navoneel/brain-mri-images-for-brain-tumor-detection
# fernando2rad/brain-tumor-mri-images-44c
```

Copy the dataset identifier (e.g., `masoudnickparvar/brain-tumor-mri-dataset`) and use it in your `config/train.yaml`.

#### Downloading Datasets

**Download directly with CLI:**

```bash
# Download a dataset
kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
```



#### Useful CLI Commands Reference

| Command | Description | Example |
|---------|-------------|---------|
| `kaggle datasets list -s "keyword"` | Search datasets | `kaggle datasets list -s "MRI"` |
| `--sort-by [method]` | Sort results | `--sort-by votes` |
| `-p [page]` | Pagination | `-p 2` |
| `kaggle datasets download -d [dataset]` | Download dataset | `kaggle datasets download -d owner/dataset` |
| `kaggle datasets files [dataset]` | List files without downloading | `kaggle datasets files owner/dataset` |
| `kaggle datasets metadata [dataset]` | Get dataset info | `kaggle datasets metadata owner/dataset` |

**Sort methods:**
- `hottest` - Trending datasets (default)
- `votes` - Most upvoted (most popular)
- `updated` - Recently updated
- `active` - Recently discussed
- `published` - Recently published

### Method 3: Browse by Category

Visit specific categories:
- **Sports:** https://www.kaggle.com/datasets?topic=sports
- **Medicine:** https://www.kaggle.com/datasets?topic=health
- **Computer Vision:** https://www.kaggle.com/datasets?topic=computer-vision
- **NLP:** https://www.kaggle.com/datasets?topic=nlp

---

## Kaggle Dataset Downloader
In your project, use this script, this create symlinks to your project data directory:

```python
import yaml
from utility.file_utils import resource_path
import pathlib
import os

try:
    import kagglehub
    KAGGLEHUB_AVAILABLE = True
except ImportError:
    KAGGLEHUB_AVAILABLE = False
    kagglehub = None


if not KAGGLEHUB_AVAILABLE:
    raise ImportError(
        "kagglehub is required for automatic dataset download. "
        "Install it with: pip install kagglehub"
    )
else:
    print(f"kagglehub is available: {kagglehub.__version__}")


config_path = resource_path("../config/train.yaml")

config_path_absolute = pathlib.Path(config_path).resolve()
if config_path_absolute.exists():
    print(f"Reading config from{config_path_absolute}")
else:
    print(f"No config found at {config_path_absolute}")
    exit()


with open(config_path_absolute) as f:
    cfg = yaml.load(f, Loader=yaml.SafeLoader)


print(cfg["data"]["kaggle_dataset"])
print(cfg["data"]["path"])


"""
Download the dataset from Kaggle if it doesn't already exist.
Checks if train and val directories exist and have content before downloading.

Uses KAGGLE_USERNAME and KAGGLE_KEY environment variables for authentication.
"""


try:
    print(f"\nAttempting to download dataset: {cfg['data']['kaggle_dataset']}")
    dataset_path = kagglehub.dataset_download(cfg["data"]["kaggle_dataset"])

    print(f"\n{'='*70}")
    print(f"✅ Dataset downloaded successfully")
    print(f"{'='*70}")
    print(f"  Source (Kaggle cache): {dataset_path}")

    # Create symlink from kaggle_structure data directory to Kaggle cache
    # Resolve path relative to kaggle_structure directory (parent of config directory)
    kaggle_structure_root = config_path_absolute.parent.parent
    target_path = (kaggle_structure_root / cfg["data"]["path"]).resolve()
    source_path = pathlib.Path(dataset_path)

    # Remove existing directory/symlink if it exists
    if target_path.exists() or target_path.is_symlink():
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.is_dir():
            print(f"  ⚠️  Target directory exists, skipping symlink creation")
            print(f"  Target (Project dir):  {target_path}")
            print(f"{'='*70}")
        else:
            target_path.unlink()

    if not target_path.exists():
        # Create parent directories if needed
        target_path.parent.mkdir(parents=True, exist_ok=True)

        # Create symlink
        target_path.symlink_to(source_path)
        print(f"  Target (Project dir):  {target_path}")
        print(f"{'='*70}")
        print(f"\n✅ Symlink created successfully")
        print(f"   {target_path} -> {source_path}")
        print(f"   (no disk space wasted)")
        print(f"{'='*70}")

except Exception as e:
    error_msg = (
        f"Failed to download or organize dataset: {e}\n"
        f"\nAuthentication options:\n"
        f"  - RunPod/Docker: Set KAGGLE_USERNAME and KAGGLE_KEY environment variables\n"
        f"  - Local: Use kagglehub.login() or create ~/.kaggle/kaggle.json\n"
        f"  - Or ensure dataset is already downloaded to: {cfg['data']['path']}"
    )
    raise RuntimeError(error_msg) from e
```

## What This Does

This script:
1. Reads dataset configuration from `config/train.yaml`
2. Downloads the specified Kaggle dataset using `kagglehub` (stores in cache: `~/.cache/kagglehub/`)
3. Creates a symlink from `kaggle_structure/data/train` to the downloaded dataset
4. Works both locally (using `kaggle.json`) and on RunPod (using environment variables)

**Benefits:**
- No duplicate data - uses symlinks
- Fast downloads - leverages Kaggle's cachedata
- Works seamlessly in local and cloud environments



---

## Running in Kaggle Notebooks

When running your code in a Kaggle notebook, the environment and paths are different from local/RunPod setups. Here's how to handle it properly.


### Environment Detection

Kaggle notebooks have specific paths and don't require authentication:

In [7]:
import os
import warnings
from tqdm import tqdm
import time
import kagglehub

warnings.filterwarnings('ignore')



# =============================================================================
# STEP 1: DATA DOWNLOAD
# =============================================================================
print("\n📥 STEP 1: Downloading dataset from KaggleHub...")
try:
    path = kagglehub.dataset_download("orvile/brain-cancer-mri-dataset")
    print("Path to dataset files:", path)
    data_path = path
except Exception as e:
    print(f"Error downloading dataset: {e}")
    print("Using local data path...")
    data_path = "data/brain-cancer/"

# Check if data exists
if not os.path.exists(data_path):
    print(f"Error: Data path '{data_path}' does not exist!")
    print("Please ensure the dataset is downloaded or available at the specified path.")


📥 STEP 1: Downloading dataset from KaggleHub...
Path to dataset files: /home/behnam/.cache/kagglehub/datasets/orvile/brain-cancer-mri-dataset/versions/2



## Kaggle Hardware & GPU

When running code in Kaggle notebooks or competitions, be aware of hardware constraints.

### Available GPUs

| GPU | VRAM | Compute Capability | Best For |
|-----|------|-------------------|----------|
| **NVIDIA Tesla P100** | 16 GB | 6.0 | General deep learning, FP32 |
| **NVIDIA Tesla T4** | 16 GB | 7.5 | Mixed precision (FP16), inference |

> **Note:** You **cannot choose** which GPU you get. Kaggle assigns P100 or T4 randomly based on availability. Both have **16GB VRAM** - design your code for this limit.

### Checking Which GPU You Got

```python
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_props.total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {gpu_props.major}.{gpu_props.minor}")
    
    if "P100" in gpu_name:
        print("→ Good for FP32 training")
    elif "T4" in gpu_name:
        print("→ Excellent for FP16/mixed precision (3-4x faster)")
else:
    print("No GPU available")
```

Or use:
```bash
!nvidia-smi
```

### GPU Comparison

| Aspect | P100 | T4 |
|--------|------|-----|
| **FP32 performance** | 9.3 TFLOPS (better) | 8.1 TFLOPS |
| **FP16 performance** | 18.7 TFLOPS | 65 TFLOPS (3-4x faster!) |
| **Memory bandwidth** | 732 GB/s (better) | 320 GB/s |
| **Power** | 250W | 70W (more efficient) |

**Bottom line:** Use mixed precision (FP16) - T4 will be much faster, P100 still works fine.

### System Resources

| Resource | Limit |
|----------|-------|
| **GPU Time** | ~30 hours/week |
| **CPU Cores** | 4 cores |
| **RAM** | 16-30 GB |
| **Disk Space** | ~73 GB (temporary) |
| **Session Time** | 9-12 hours max |
| **Internet** | Enabled (disabled in competitions) |

### Critical Limitations

1. 16 GB VRAM (Most Common Bottleneck)
2. Session Timeout (9-12 hours)
3. Disk Space (~73 GB)



---

## GPU Optimization Best Practices

Strategies to maximize GPU utilization and avoid out-of-memory errors on Kaggle's 16GB GPUs.

1. Use Mixed Precision Training (FP16)
2. Gradient Accumulation
3. Monitor GPU Memory
4. Find Optimal Batch Size Dynamically
5. Clear Unused Variables & Cache
6. Gradient Checkpointing
7. Use Efficient Data Loading
8. Reduce Model Precision Strategically
9. GPU-Agnostic Code Pattern

## Uploading and Creating your Own Dataset
[rvp group slam dataset](https://rvp-group.net/slam-dataset.html)

Refs [1](https://www.kaggle.com/docs/api#installation)